# FlashNystrom: the paper's training sweep, on Colab

Runs **every** training experiment behind the paper under a hard **20 hour**
cap, on one A100. Run the cells top to bottom.

Two things make this survivable on Colab, where runtimes get reset:

1. **Results live on Google Drive**, not the ephemeral disk. A reset loses at
   most the single run in flight.
2. **Everything is resumable.** If the runtime dies, re-run the notebook top to
   bottom; finished cells are skipped via their result JSON and the sweep
   continues where it stopped. The 20h cap counts only *this* session, so
   check the elapsed total yourself across resumes.

The cap covers setup as well as training: section 6 subtracts the time already
spent so the total stays under 20h.

Expect roughly 14h of training under `--preset paper12`, so there is real
headroom. If the budget runs out anyway, jobs run highest-value-first and the
driver stops cleanly, so what is missing is the least load-bearing experiment.

## 0. Session clock and GPU

Everything downstream measures against `T0`.

In [ ]:
import time, torch, os
T0 = time.time()
BUDGET_H = 20.0

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> A100."
p = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability()
print(f"{p.name} | sm_{cc[0]}{cc[1]} | {p.total_memory/1e9:.0f} GB | torch {torch.__version__}")
assert cc >= (8, 0), "flash_nystrom needs compute capability >= 8.0 (A100/L4/H100)"
if "A100" not in p.name:
    print("\n!! Not an A100. The 20h budget was costed for an A100-80GB; a smaller\n"
          "   GPU will get through less of the sweep before the cap. That is fine,\n"
          "   the ordering means you lose the least important runs first.")

## 1. Mount Drive

Results and logs go here so a runtime reset does not destroy the sweep. This is
the single most important cell in the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/flashnystrom_paper'
RUNS, LOGS = f'{DRIVE}/runs', f'{DRIVE}/logs'
os.makedirs(RUNS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
print('results ->', RUNS)

done = [f for f in os.listdir(RUNS) if f.endswith('.json')] if os.path.isdir(RUNS) else []
print(f'{len(done)} result file(s) already on Drive'
      + (' -> this is a RESUME, finished runs will be skipped' if done else ''))

## 2. Clone and build

`--no-build-isolation` is required: setup.py imports torch.

In [ ]:
%cd /content
!rm -rf FlashNystrom
!git clone --recursive -q https://github.com/athrva98/FlashNystrom.git
%cd /content/FlashNystrom
!git log --oneline -3

!pip -q install einops ninja packaging
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"
os.environ["FLASH_NYSTROM_LAX_BUILD"] = "1"
!pip install -e . --no-build-isolation 2>&1 | tail -2

import flash_nystrom
from flash_nystrom import flash_nystrom_attention
q = torch.randn(1, 2, 512, 64, device='cuda', dtype=torch.float16, requires_grad=True)
o = flash_nystrom_attention(q, q.clone(), q.clone(), num_landmarks=64); o.sum().backward()
print('flash_nystrom', flash_nystrom.__version__, '| fwd+bwd OK on this GPU')

## 3. Baseline kernels

**Both matter for fairness.** Without `flash_attn` the sliding-window arm cannot
run at all. Without `flash_bla` the linear-attention arm silently falls back to
an unfused torch path, roughly 2x slower, which is not a fair baseline.

**This cell is the slow one.** `flash_attn` has no wheel matching Colab's torch,
so it compiles from source: budget 1 to 2 hours, and it counts against the 20h
cap. `MAX_JOBS=4` keeps the build from exhausting RAM and being killed, which is
the usual failure here.

If you are short on time you can skip this cell and drop `sliding_window` from
`--arms` in section 6. You lose one of seven arms and keep everything else. Do
NOT skip it and leave the arm in: it will fail every cell it appears in.

In [ ]:
import subprocess, time
t = time.time()
os.environ["MAX_JOBS"] = "4"          # source build OOMs the runtime unbounded
!pip install flash-attn --no-build-isolation 2>&1 | tail -3
!pip install -q -e git+https://github.com/fla-org/flash-bidirectional-linear-attention.git#egg=flash_bla 2>&1 | tail -2
!pip -q install pyfaidx genomic-benchmarks

for mod, why in [("flash_attn", "sliding_window arm"),
                 ("flash_bla", "linear_attention arm (FAIRNESS: unfused fallback is not a valid baseline)"),
                 ("pyfaidx", "genomics species task"),
                 ("genomic_benchmarks", "genomics regulatory task")]:
    ok = subprocess.run(["python", "-c", f"import {mod}"], capture_output=True).returncode == 0
    print(f"  {mod:20s} {'OK' if ok else '!! MISSING':10s} {why}")
print(f"\nbaseline setup took {(time.time()-t)/60:.0f} min")

## 4. Reference genomes

About 1.5 GB from Ensembl, cached on Drive so a reset does not re-download it.
Four of HyenaDNA's five species exactly; goat substitutes for hippopotamus,
which has no chromosome-level assembly and is in their own split table.

In [ ]:
GENOMES = f'{DRIVE}/genomes'
os.makedirs(GENOMES, exist_ok=True)

# Verify the EXACT files the split table needs. Counting files is not enough:
# download_genomes.py defaults to chroms_per_split=4 (40 .fa), .fai indexes
# inflate any count, and a partially finished download then looks complete
# and fails later inside the sweep instead of here.
import sys
sys.path.insert(0, '/content/FlashNystrom')
from benchmarks.genomics_data import SPECIES_CHROMOSOME_SPLITS, DEFAULT_SPECIES

CHROMS_PER_SPLIT = 4
need = [(s, c) for s in DEFAULT_SPECIES
        for sp in ('train', 'test')
        for c in SPECIES_CHROMOSOME_SPLITS[s][sp][:CHROMS_PER_SPLIT]]
have = lambda: [(s, c) for s, c in need
                if not os.path.exists(f'{GENOMES}/{s}/{c}.fa')]
missing = have()
print(f'{len(need) - len(missing)}/{len(need)} chromosome files present')

if missing:
    print(f'fetching {len(missing)} missing (~1.5GB total, cached on Drive)')
    !python benchmarks/download_genomes.py --out {GENOMES} --chroms_per_split {CHROMS_PER_SPLIT}
    missing = have()

if missing:
    print(f'STILL MISSING {len(missing)}: {missing[:6]}')
    print('The species stage WILL fail. Re-run this cell before section 6.')
else:
    print('genomes complete')

## 5. Smoke test

Ten minutes that protect the next twenty hours. Runs every stage and every
arm with budgets cut to near zero. Its accuracies are meaningless by
construction and go to a separate directory.

**If this reports failures other than a missing optional kernel, stop and fix
them before section 6.**

In [ ]:
!python run_all_paper_experiments.py --smoke --species_dir {GENOMES} 2>&1 | tail -25

## 6. The sweep

Hard-capped: the budget passed to the driver is 20h **minus the time already
spent above**, so setup does not eat into training and the session total stays
under 20h.

Jobs run highest-value-first. If the cap is reached, the driver stops launching
and says exactly what it skipped; re-running this cell later continues.

Run this cell and leave the tab open.

In [ ]:
spent_h = (time.time() - T0) / 3600
remaining = max(0.5, BUDGET_H - spent_h)
print(f'setup used {spent_h:.2f}h -> training budget {remaining:.2f}h')

# Built as ONE string and run as a single !{cmd}. IPython does not interpolate
# {var} across backslash-continued ! lines, so a multi-line command silently
# passes the literal text "{remaining:.2f}" to argparse.
cmd = (f"python run_all_paper_experiments.py --preset paper12 "
       f"--budget_hours {remaining:.2f} --out {RUNS} --species_dir {GENOMES} "
       f"2>&1 | tee {LOGS}/sweep.log | tail -80")
print(cmd, flush=True)
!{cmd}


## 7. Results

Prints the tables and copies logs to Drive.

In [ ]:
!cp -r logs/* {LOGS}/ 2>/dev/null; echo 'logs -> Drive'

print('=' * 70, '\nMQAR\n', '=' * 70)
!python -m paper.mqar.paper_sweep --collect_only --out {RUNS}/mqar 2>&1 | tail -20

for task in ('genomic_benchmarks', 'species', 'repeat'):
    print('\n' + '=' * 70, f'\nGENOMICS: {task}\n', '=' * 70)
    !python benchmarks/run_genomics.py --task {task} --collect_only --out {RUNS}/genomics 2>&1 | tail -15

print('\n' + '=' * 70, '\nVISION\n', '=' * 70)
import json, glob
for f in sorted(glob.glob(f'{RUNS}/vision_*.json')):
    r = json.load(open(f))
    tag = os.path.basename(f)[7:-5]
    accs = '  '.join(f"{x['label']}={x['test_acc']:.1f}" for x in r['results'])
    print(f"  {tag:28s} N={r.get('n_tokens','?'):>6}  {accs}")

## 8. If the runtime reset

Re-run the notebook top to bottom. Sections 1-4 are cached on Drive and return
quickly, section 5 is ten minutes, and section 6 resumes the sweep where it
stopped. Track your own cumulative hours: `T0` resets each session, so the
20h cap applies per session, not across resumes.

To see what is still outstanding without running anything:

```
!python run_all_paper_experiments.py --preset paper12 --out {RUNS} --list
```